# Graph t-SNE from reconstructed canonical attention

This notebook works **directly on the reconstructed canonical split** through `ResearchDataset`; it does not require a separately built `graphs/` cache.

For each sample, `ResearchSample.attention_edges()` decodes the canonical response-attention CSR into sparse `(layer, head, source, target, weight)` edges. Repeated source→target relations across channels are merged into mean-channel relation weights, 12 token-level structural signals are computed, and `temporal_summary` converts them to a fixed 36-D sample descriptor (`mean/std/slope`). Labels are read only after t-SNE is fitted and are used only for visualization and optional post-hoc interpretation.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "research_dataset.py").is_file():
    raise RuntimeError("Run this notebook from the repository root or its notebooks directory.")
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

from descriptors import temporal_summary
from research_dataset import LabelStore, ResearchDataset

DATA_ROOT = Path(
    "/share/home/tm902089733300000/a903202310/lys/data/RAGTruth/"
    "model_traces/llama31_8b/test"
)

DEVICE = "cpu"
VERIFY_HASHES = False  # Set True for per-sample SHA256 verification; slower on a full split.
MAX_SAMPLES = None     # e.g. 1000 for a deterministic quick run; labels are not used for sampling.
RANDOM_STATE = 0
SAVE_DIR = None        # e.g. Path("outputs/graph_tsne_raw")


## Direct structural features from canonical sparse attention

No graph cache is loaded here. The canonical attention archive already contains the sparse response-query relations needed for structural analysis. The per-token features cover routing strength, prompt-vs-history dependence, concentration, locality, unique relation degree/density, and channel-level sparsity.


In [ ]:
RAW_GRAPH_FEATURE_NAMES = (
    "incoming_mass",
    "prompt_mass_share",
    "normalized_entropy",
    "history_lag",
    "in_degree",
    "prompt_degree",
    "history_degree",
    "in_density",
    "prompt_density",
    "history_density",
    "history_edge_share",
    "channel_edge_density",
)


def raw_attention_graph_features(attention, edges):
    """Return [response_tokens, 12] graph features directly from canonical attention CSR.

    The canonical archive stores one sparse edge per
    (layer, head, response target, earlier source) above attention_floor.
    For graph-level structural statistics, repeated source->target relations
    across channels are merged by mean-channel weight.
    """
    response_idx = attention.response_idx
    response_count = attention.num_response_tokens
    num_nodes = attention.num_tokens
    num_channels = attention.num_channels

    features = torch.zeros((response_count, len(RAW_GRAPH_FEATURE_NAMES)), dtype=torch.float32)
    if edges["weight"].numel() == 0:
        return features

    source = edges["source"].long().cpu()
    target = edges["target"].long().cpu()
    weight = edges["weight"].float().cpu()
    rows = target - response_idx

    if bool(((rows < 0) | (rows >= response_count)).any()):
        raise ValueError("decoded attention edge targets must be response tokens")
    if bool(((source < 0) | (source >= target)).any()):
        raise ValueError("decoded attention edges must point to earlier tokens")

    # Merge the same source->target relation across layer/head channels.
    pair_key = rows * num_nodes + source
    unique_key, inverse = torch.unique(pair_key, sorted=True, return_inverse=True)
    pair_weight = torch.zeros(unique_key.numel(), dtype=torch.float32)
    pair_weight.index_add_(0, inverse, weight)
    pair_weight /= float(num_channels)

    pair_rows = torch.div(unique_key, num_nodes, rounding_mode="floor")
    pair_source = unique_key.remainder(num_nodes)
    pair_prompt = pair_source < response_idx
    pair_history = ~pair_prompt

    total_mass = torch.zeros(response_count)
    total_mass.index_add_(0, pair_rows, pair_weight)

    prompt_mass = torch.zeros(response_count)
    prompt_mass.index_add_(0, pair_rows[pair_prompt], pair_weight[pair_prompt])
    nonempty = total_mass > 0
    prompt_share = torch.zeros(response_count)
    prompt_share[nonempty] = prompt_mass[nonempty] / total_mass[nonempty]

    in_degree = torch.bincount(pair_rows, minlength=response_count).float()
    prompt_degree = torch.bincount(pair_rows[pair_prompt], minlength=response_count).float()
    history_degree = torch.bincount(pair_rows[pair_history], minlength=response_count).float()

    probabilities = pair_weight / total_mass[pair_rows]
    entropy = torch.zeros(response_count)
    entropy.index_add_(0, pair_rows, -probabilities * probabilities.log())
    normalized_entropy = torch.zeros(response_count)
    multiple = in_degree > 1
    normalized_entropy[multiple] = entropy[multiple] / in_degree[multiple].log()

    history_mass = torch.zeros(response_count)
    history_mass.index_add_(0, pair_rows[pair_history], pair_weight[pair_history])
    history_lag_mass = torch.zeros(response_count)
    if bool(pair_history.any()):
        history_target = response_idx + pair_rows[pair_history]
        lag = (history_target - pair_source[pair_history]).float()
        history_lag_mass.index_add_(
            0,
            pair_rows[pair_history],
            pair_weight[pair_history] * lag / max(response_count - 1, 1),
        )
    history_lag = torch.zeros(response_count)
    has_history_mass = history_mass > 0
    history_lag[has_history_mass] = (
        history_lag_mass[has_history_mass] / history_mass[has_history_mass]
    )

    response_position = torch.arange(response_count, dtype=torch.float32)
    absolute_target = response_idx + response_position
    in_density = in_degree / absolute_target.clamp_min(1.0)
    prompt_density = prompt_degree / float(max(response_idx, 1))
    history_density = torch.zeros(response_count)
    has_history = response_position > 0
    history_density[has_history] = history_degree[has_history] / response_position[has_history]

    history_edge_share = torch.zeros(response_count)
    has_edges = in_degree > 0
    history_edge_share[has_edges] = history_degree[has_edges] / in_degree[has_edges]

    # Channel-level sparsity: retained (layer, head, source) entries among all
    # possible earlier-token entries for each response query.
    channel_degree = torch.bincount(rows, minlength=response_count).float()
    channel_edge_density = channel_degree / (
        float(num_channels) * absolute_target.clamp_min(1.0)
    )

    columns = (
        total_mass,
        prompt_share,
        normalized_entropy,
        history_lag,
        in_degree,
        prompt_degree,
        history_degree,
        in_density,
        prompt_density,
        history_density,
        history_edge_share,
        channel_edge_density,
    )
    features = torch.stack(columns, dim=1)
    if not bool(torch.isfinite(features).all()):
        raise ValueError("raw graph features must be finite")
    return features


def fit_tsne(matrix, random_state=0):
    if len(matrix) < 3:
        raise ValueError("t-SNE analysis needs at least three samples.")

    scaled = StandardScaler().fit_transform(matrix)
    if scaled.shape[1] > 50:
        n_components = min(50, scaled.shape[0], scaled.shape[1])
        scaled = PCA(n_components=n_components, random_state=random_state).fit_transform(scaled)

    perplexity = min(30.0, max(2.0, (len(scaled) - 1) / 3.0))
    perplexity = min(perplexity, len(scaled) - 1.0)
    coordinates = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        max_iter=1000,
        random_state=random_state,
    ).fit_transform(scaled)
    return coordinates, perplexity


## Build one fixed graph descriptor per sample and fit t-SNE

`MAX_SAMPLES` truncates by dataset order only; labels are not consulted. Each sample is loaded and summarized independently, so the notebook does not keep full attention tensors for the whole split in memory.


In [ ]:
dataset = ResearchDataset(DATA_ROOT, device=DEVICE, verify_hashes=VERIFY_HASHES)
sample_ids = dataset.sample_ids if MAX_SAMPLES is None else dataset.sample_ids[:MAX_SAMPLES]

descriptor_rows = []
for sample_id in tqdm(sample_ids, desc="raw attention graph descriptors"):
    sample = dataset[sample_id]
    attention = sample.attention()
    edges = sample.attention_edges(attention=attention)
    token_features = raw_attention_graph_features(attention, edges)
    descriptor_rows.append(temporal_summary(token_features).cpu().numpy())

descriptor_matrix = np.stack(descriptor_rows)
descriptor_names = np.asarray(
    [
        f"{stat}_{name}"
        for stat in ("mean", "std", "slope")
        for name in RAW_GRAPH_FEATURE_NAMES
    ]
)

coordinates, perplexity = fit_tsne(descriptor_matrix, RANDOM_STATE)

print(f"samples: {len(sample_ids)}")
print(f"token graph features: {len(RAW_GRAPH_FEATURE_NAMES)}")
print(f"graph descriptor shape: {descriptor_matrix.shape}")
print(f"t-SNE perplexity: {perplexity:.2f}")


## Evaluation-only coloring

The embedding above is label-free. `positive_runs` is reduced to a sample-level indicator only for plotting: any positive response span → hallucinated sample.


In [ ]:
# Labels are intentionally loaded only after descriptor extraction and t-SNE fitting.
label_store = LabelStore(DATA_ROOT / "labels.jsonl")
labels = np.asarray(
    [int(bool(label_store.positive_runs(sample_id))) for sample_id in sample_ids],
    dtype=np.int64,
)

figure, axis = plt.subplots(figsize=(7.5, 6.0), constrained_layout=True)
for value, name in ((0, "Correct"), (1, "Hallucinated")):
    mask = labels == value
    axis.scatter(
        coordinates[mask, 0],
        coordinates[mask, 1],
        s=24,
        alpha=0.72,
        label=f"{name} (n={int(mask.sum())})",
    )

axis.set(
    title="Raw attention-graph t-SNE",
    xlabel="t-SNE 1",
    ylabel="t-SNE 2",
)
axis.legend(frameon=False)
axis.grid(alpha=0.15)
plt.show()


In [ ]:
# Optional post-hoc interpretation: which graph descriptors differ most by sample label?
# This cell does not affect the t-SNE coordinates above.
if len(np.unique(labels)) == 2:
    standardized = StandardScaler().fit_transform(descriptor_matrix)
    correct_mean = standardized[labels == 0].mean(axis=0)
    hallucinated_mean = standardized[labels == 1].mean(axis=0)
    delta = hallucinated_mean - correct_mean
    order = np.argsort(np.abs(delta))[::-1][:12]

    print("Largest standardized Hallucinated - Correct descriptor differences:")
    for index in order:
        print(f"{descriptor_names[index]:>36s}  {delta[index]: .4f}")


In [ ]:
if SAVE_DIR is not None:
    SAVE_DIR = Path(SAVE_DIR)
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        SAVE_DIR / "raw_attention_graph_tsne.npz",
        sample_id=np.asarray(sample_ids),
        descriptor=descriptor_matrix,
        descriptor_name=descriptor_names,
        coordinates=coordinates,
        label=labels,
    )

    figure.savefig(
        SAVE_DIR / "raw_attention_graph_tsne.png",
        dpi=200,
        bbox_inches="tight",
    )
